# MLPと誤差逆伝播をnumpyで実装する(演習)

解説資料は `research-handbook/deep-learning/nn-basics.md`・`backprop.md`。
このnotebookは**直接編集せず**、自分の卒研repoにコピーして使うこと(手順は `technical-handbook/colab/use-github-repo.md`)。

「---- ここから課題 ----」の区間を埋めながら上から順に実行する。
解答付き版は `research-handbook/notebooks/mlp_backprop_solution.ipynb` にあるが、まず自力で取り組むこと。

PyTorchの `loss.backward()` が中で何をしているかを、一度は自分の手で書いて理解する。これが `dqn_cartpole.ipynb` 以降の深層RLのコードを読む土台になる。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 線形分離できないデータ: 同心円状の2クラス
def make_circles(n=400, seed=0):
    rng = np.random.default_rng(seed)
    r = np.concatenate([rng.normal(1.0, 0.15, n // 2), rng.normal(2.2, 0.15, n - n // 2)])
    th = rng.uniform(0, 2 * np.pi, n)
    X = np.stack([r * np.cos(th), r * np.sin(th)], axis=1)
    t = np.concatenate([np.zeros(n // 2), np.ones(n - n // 2)])
    return X, t

X_train, t_train = make_circles(400, seed=0)
X_test,  t_test  = make_circles(400, seed=1)
plt.figure(figsize=(4, 4))
plt.scatter(*X_train.T, c=t_train, cmap="coolwarm", s=10); plt.axis("equal"); plt.show()

## 線形モデルの限界

まずロジスティック回帰(`machine-learning/classification.md`)で試します。同心円は直線1本では分けられないので、原理的にaccuracy 0.5前後にしかなりません。**基底関数を固定せず学習する**のがMLPの本質です(`nn-basics.md`)。

In [ ]:
# まずロジスティック回帰(線形モデル)で試す → 原理的に無理なことを確認
def sigmoid(a):
    return 1 / (1 + np.exp(-np.clip(a, -500, 500)))   # クリップでオーバーフロー回避

Phi = np.hstack([np.ones((len(X_train), 1)), X_train])
w = np.zeros(3)
for _ in range(2000):
    w -= 0.01 * Phi.T @ (sigmoid(Phi @ w) - t_train)
acc_lin = np.mean((sigmoid(Phi @ w) > 0.5) == t_train)
print(f"ロジスティック回帰のtrain accuracy = {acc_lin:.3f}(ほぼコイン投げ)")
# 直線1本では同心円は分けられない → 基底関数を「学習」するMLPの出番

## 課題1: 順伝播

2層MLP(入力2 → 隠れ32 → 出力1):

$$z^{(1)} = X W_1 + b_1, \quad a^{(1)} = \mathrm{ReLU}(z^{(1)}), \quad z^{(2)} = a^{(1)} W_2 + b_2, \quad y = \sigma(z^{(2)})$$

逆伝播で使うので中間量($X, z^{(1)}, a^{(1)}$)も保存しておきます。初期化はHe初期化(`optimization-regularization.md`)。

In [ ]:
def init_params(sizes, seed=0):
    """He初期化(optimization-regularization.md 参照)"""
    rng = np.random.default_rng(seed)
    params = {}
    for l in range(len(sizes) - 1):
        params[f"W{l+1}"] = rng.normal(0, np.sqrt(2.0 / sizes[l]), (sizes[l], sizes[l + 1]))
        params[f"b{l+1}"] = np.zeros(sizes[l + 1])
    return params

def relu(z):
    return np.maximum(0, z)

def forward_pass(X, params):
    """課題1: 2層MLPの順伝播。
    z1 = X W1 + b1 → a1 = ReLU(z1) → z2 = a1 W2 + b2 → y = sigmoid(z2)
    逆伝播で使う中間量(cache)も返す"""
    # ---- ここから課題1 ----
    # nn-basics.md の行列表記の通り。y は .ravel() で1次元にする
    z1 = None
    a1 = None
    z2 = None
    y = None
    # ---- ここまで ----
    cache = {"X": X, "z1": z1, "a1": a1}
    return y, cache

params = init_params([2, 32, 1])
y, cache = forward_pass(X_train, params)
print("出力の形:", y.shape, " 出力の範囲: [", y.min().round(3), ",", y.max().round(3), "]")

## 課題2: 逆伝播と勾配チェック

`backprop.md` の再帰式をそのまま実装します。シグモイド+交差エントロピーの組では出力層の誤差が $\delta^{(2)} = y - t$ になります(導出は `backprop.md`)。隠れ層へは

$$\delta^{(1)} = (\delta^{(2)} W_2^\top) \odot \mathrm{ReLU}'(z^{(1)}), \qquad \frac{\partial E}{\partial W_l} = a^{(l-1)\top} \delta^{(l)}$$

**実装したら必ず数値微分と比較**します。誤差が $10^{-7}$ より大きければどこかにバグがあります。

In [ ]:
def cross_entropy(y, t):
    eps = 1e-12
    return -np.mean(t * np.log(y + eps) + (1 - t) * np.log(1 - y + eps))

def backward_pass(y, t, params, cache):
    """課題2の解答: 逆伝播。backprop.md の δ の再帰式に対応。
    シグモイド+交差エントロピーなので出力層の δ は (y - t) になる(導出は backprop.md)"""
    N = len(t)
    X, z1, a1 = cache["X"], cache["z1"], cache["a1"]
    grads = {}
    # ---- ここから課題2 ----
    # (1) 出力層: delta2 = (y - t).reshape(-1, 1) / N
    # (2) grads["W2"] = a1^T delta2, grads["b2"] = delta2 の列和
    # (3) 隠れ層: delta1 = (delta2 W2^T) * ReLU'(z1)。ReLU'(z1) は (z1 > 0)
    # (4) grads["W1"], grads["b1"] も同様に
    # 完成したら下の勾配チェックで必ず検算すること


    # ---- ここまで ----
    return grads

# 勾配チェック: 数値微分と比較(REINFORCE・ヤコビアンと同じ検算の習慣)
def numerical_grad(X, t, params, key, eps=1e-6):
    g = np.zeros_like(params[key])
    it = np.nditer(params[key], flags=["multi_index"])
    while not it.finished:
        idx = it.multi_index
        orig = params[key][idx]
        params[key][idx] = orig + eps
        lp = cross_entropy(forward_pass(X, params)[0], t)
        params[key][idx] = orig - eps
        lm = cross_entropy(forward_pass(X, params)[0], t)
        params[key][idx] = orig
        g[idx] = (lp - lm) / (2 * eps)
        it.iternext()
    return g

small = init_params([2, 4, 1], seed=1)
Xs, ts = X_train[:20], t_train[:20]
ys, cs = forward_pass(Xs, small)
gs = backward_pass(ys, ts, small, cs)
for key in ["W1", "b1", "W2", "b2"]:
    gn = numerical_grad(Xs, ts, small, key)
    print(f"{key}: 解析勾配と数値勾配の最大誤差 = {np.abs(gs[key] - gn).max():.2e}")

## 課題3: 学習ループ

定型は forward → loss → backward → update の4段(PyTorchの5行定型 `pytorch-intro.md` と同じ構造)。損失が単調に下がり、テストaccuracyがロジスティック回帰を大きく超えることを確認します。

In [ ]:
def train_mlp(X, t, hidden=32, lr=0.5, n_iters=2000, seed=0):
    """課題3の解答: 勾配降下による学習ループ(定型: forward → loss → backward → update)"""
    params = init_params([X.shape[1], hidden, 1], seed=seed)
    history = []
    for i in range(n_iters):
        # ---- ここから課題3 ----
        # forward → loss(記録) → backward → 全パラメータを勾配降下で更新
        pass
        # ---- ここまで ----
    return params, history

params, history = train_mlp(X_train, t_train)
acc_train = np.mean((forward_pass(X_train, params)[0] > 0.5) == t_train)
acc_test  = np.mean((forward_pass(X_test,  params)[0] > 0.5) == t_test)
print(f"MLP: train accuracy = {acc_train:.3f}, test accuracy = {acc_test:.3f}")

# 決定境界の可視化
gx, gy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
grid = np.stack([gx.ravel(), gy.ravel()], axis=1)
zz = forward_pass(grid, params)[0].reshape(gx.shape)
plt.figure(figsize=(9, 3.5))
plt.subplot(1, 2, 1); plt.plot(history); plt.xlabel("iteration"); plt.ylabel("loss")
plt.subplot(1, 2, 2)
plt.contourf(gx, gy, zz, levels=20, cmap="coolwarm", alpha=0.6)
plt.scatter(*X_train.T, c=t_train, cmap="coolwarm", s=8, edgecolors="k", linewidths=0.2)
plt.axis("equal"); plt.title("decision boundary")
plt.tight_layout(); plt.show()

## 発展: 隠れ層の幅

幅を1, 2, 4, 32と変えると、円形の境界を表現するのに必要な「学習された基底関数」の数が見えてきます。

In [ ]:
# 発展: 隠れ層の幅と表現力
for h in [1, 2, 4, 32]:
    p, _ = train_mlp(X_train, t_train, hidden=h)
    acc = np.mean((forward_pass(X_test, p)[0] > 0.5) == t_test)
    print(f"hidden={h}: test accuracy = {acc:.3f}")
# 幅1〜2では円形の境界を表現できない(基底関数が足りない)

## 発展課題

1. 学習率を10倍/100分の1にして損失曲線の変化を観察する(`../reinforcement-learning/hyperparameter.md` の症状表)
2. ReLUをsigmoidに変えて学習の速さを比較する(勾配消失。`backprop.md`)
3. 同じネットワークをPyTorchで書き直し、`loss.backward()` の勾配と自作逆伝播の勾配が一致することを確認する(`pytorch-intro.md`)

## まとめ

- MLPは「基底関数を学習する」モデル。線形モデルで無理な問題が解ける
- 逆伝播はδの再帰式+勾配チェックで確実に実装できる
- forward → loss → backward → update の定型はPyTorchでもDQNでも同じ
- 次は `deep-learning/optimization-regularization.md`(Adam・正則化)→ `pytorch-intro.md` → 深層RL(`../reinforcement-learning/dqn.md`)へ